# Evaluasi Dampak Slang Normalization dan N-gram Feature terhadap Performa Model Boosting
## (XGBoost, LightGBM, CatBoost, AdaBoost) Dalam Deteksi Spam

**Authors:** Alexander Christian Suryanto Linggodigdo, Fanny Octaviana Pangestu, Howard Frelindo Goh  
**Institution:** Computer Science – BINUS University

**Dataset:** [Deysi/spam-detection-dataset](https://huggingface.co/datasets/Deysi/spam-detection-dataset) (Hugging Face) — 10,900 messages, labels `spam` / `not_spam`, with official train (8,180) and test (2,720) splits.

---
**How to run:**
1. Run **Cell 1** to install packages (restart runtime if prompted)
2. Run **Cell 2** to download the dataset from Hugging Face (no manual upload needed)
3. Run all remaining cells in order (**Runtime → Run all**)


In [ ]:
# ── Cell 1: Install packages ──────────────────────────────────────────────────
!pip install -q xgboost lightgbm catboost contractions datasets
print('Installation complete. If you see a restart prompt, restart and then continue from Cell 2.')


In [ ]:
# ── Cell 2: Download dataset from Hugging Face ───────────────────────────────
# Deysi/spam-detection-dataset
#   columns : text (string), label ('spam' / 'not_spam')
#   splits  : train = 8,180 rows, test = 2,720 rows
from datasets import load_dataset

hf_ds = load_dataset("Deysi/spam-detection-dataset")
print("Loaded splits:", {k: v.num_rows for k, v in hf_ds.items()})
print("Columns      :", hf_ds['train'].column_names)
print("Label values :", sorted(set(hf_ds['train']['label'])))


In [ ]:
# ── Cell 3: Imports ───────────────────────────────────────────────────────────
import re
import warnings
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
from sklearn.preprocessing import LabelEncoder

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import contractions

warnings.filterwarnings('ignore')
nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('All imports OK.')

In [ ]:
# ── Cell 4: SMS Slang Dictionary ──────────────────────────────────────────────
SLANG_DICT = {
    # Pronouns / common short words
    "u": "you",       "ur": "your",      "r": "are",        "b": "be",
    "y": "why",       "k": "ok",         "m": "am",         "n": "and",
    "2": "to",        "4": "for",        "4u": "for you",   "b4": "before",
    "2day": "today",  "2nite": "tonight","2moro": "tomorrow","2morrow": "tomorrow",
    "2mrw": "tomorrow","4got": "forgot", "4ever": "forever","2gether": "together",
    # Expressions / reactions
    "gm": "good morning",   "gn": "good night",       "gud": "good",
    "gr8": "great",          "g8": "great",            "gr8t": "great",
    "lol": "laugh out loud", "lmao": "laugh my ass off",
    "rofl": "rolling on the floor laughing",
    "omg": "oh my god",      "omfg": "oh my god",
    "wtf": "what the hell",  "wth": "what the hell",
    "idk": "i do not know",  "imo": "in my opinion",
    "imho": "in my humble opinion", "tbh": "to be honest",
    "ngl": "not gonna lie",  "fyi": "for your information",
    "btw": "by the way",     "jk": "just kidding",
    "brb": "be right back",  "bbl": "be back later",
    "bbs": "be back soon",   "afk": "away from keyboard",
    "np": "no problem",      "yw": "you are welcome",
    "ty": "thank you",       "thx": "thanks",
    "thnx": "thanks",        "tnx": "thanks",     "tx": "thanks",
    "pls": "please",         "plz": "please",     "plx": "please",
    "sry": "sorry",          "srry": "sorry",
    "nvm": "never mind",     "nvr": "never",      "evr": "ever",
    # Question / reference words
    "wut": "what",  "wat": "what",  "wats": "what is",
    "abt": "about", "hw": "how",    "hru": "how are you",
    "wen": "when",  "whr": "where", "wer": "where",
    "der": "there", "dere": "there",
    "dis": "this",  "dat": "that",  "dats": "that is",
    "d": "the",     "da": "the",    "de": "the",
    "dem": "them",  "dey": "they",
    # Time / convenience
    "asap": "as soon as possible", "atm": "at the moment",
    "nite": "night",  "nyt": "night",
    "tmrw": "tomorrow", "tmr": "tomorrow", "tmo": "tomorrow",
    "l8r": "later",   "l8": "late",  "ltr": "later",
    "ttyl": "talk to you later",    "ttys": "talk to you soon",
    "tc": "take care",
    "cya": "see you", "cu": "see you", "cul8r": "see you later",
    "b4n": "bye for now",
    # Colloquial contractions (without apostrophes — after punctuation removal)
    "wanna": "want to",  "gonna": "going to",  "gotta": "got to",
    "kinda": "kind of",  "sorta": "sort of",
    "dunno": "do not know", "lemme": "let me",
    "gimme": "give me",     "cmon": "come on",
    # Alphanumeric slang
    "w8": "wait",   "h8": "hate",  "m8": "mate",
    "d8": "date",   "f8": "fate",  "sk8": "skate",
    "ne1": "anyone",  "ne": "any",  "neway": "anyway",
    "nething": "anything",
    "evry1": "everyone", "every1": "everyone",
    "som1": "someone",   "sum1": "someone", "no1": "no one",
    # SMS / messaging
    "txt": "text",   "msg": "message",  "msgs": "messages",
    "mob": "mobile", "mobi": "mobile",
    "rply": "reply", "rpl": "reply",    "stp": "stop",
    # Informal negations / modals (post punctuation-removal, no apostrophe)
    "dont": "do not",       "doesnt": "does not",  "cant": "cannot",
    "wont": "will not",     "couldnt": "could not","wouldnt": "would not",
    "shouldnt": "should not","im": "i am",          "ive": "i have",
    "id": "i would",        "ill": "i will",
    "isnt": "is not",       "arent": "are not",    "wasnt": "was not",
    "werent": "were not",   "havent": "have not",  "hasnt": "has not",
    "hadnt": "had not",     "didnt": "did not",
    "theyre": "they are",   "theyd": "they would", "theyll": "they will",
    "theyve": "they have",  "youre": "you are",    "youd": "you would",
    "youll": "you will",    "youve": "you have",
    "hes": "he is",         "shes": "she is",      "its": "it is",
    "weve": "we have",      "wed": "we would",
    # Spam-related terms
    "wk": "week",   "wkly": "weekly", "mth": "month",
    "yr": "year",   "min": "minute",  "mins": "minutes",
    "hr": "hour",   "hrs": "hours",   "wks": "weeks",
    "acc": "account", "acct": "account", "amt": "amount",
    "cd": "could",   "gt": "got",      "hav": "have",  "hve": "have",
    "nd": "and",     "sm": "some",     "sme": "some",
    "ppl": "people", "nt": "not",      "nw": "now",   "nxt": "next",
    "smth": "something",  "sumthing": "something",
    "grt": "great",        "amzing": "amazing",
    "cust": "customer",    "custmr": "customer",
    "dlr": "dollar",       "dlrs": "dollars",
    "reciev": "receive",   "recive": "receive",  "receve": "receive",
    "reedem": "redeem",    "redeme": "redeem",
    "subscrib": "subscribe", "unsubscrib": "unsubscribe",
    "exclusiv": "exclusive", "xclusiv": "exclusive",
    "mrkt": "market",     "mkt": "market",
    "priz": "prize",
}

# Compile a single regex (longest match first to avoid partial replacements)
_slang_keys_sorted = sorted(SLANG_DICT.keys(), key=len, reverse=True)
_slang_pattern = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in _slang_keys_sorted) + r')\b'
)
print(f'Slang dictionary loaded: {len(SLANG_DICT)} entries.')

In [ ]:
# ── Cell 5: Preprocessing pipeline ───────────────────────────────────────────
# NLTK's English stopwords list contains words that are critical spam signals
# (e.g. "won", "not", "no", "only", "very") and de-apostrophised negation
# stems (e.g. "couldn", "wouldn", "haven"). Stripping them blinds the model to
# obvious cues like "you have won …".  Keep them.
_KEEP_WORDS = {
    # Negations and modifiers that NLTK strips but the model needs.
    'won', 'win', 'no', 'not', 'only', 'very', 'against', 'again',
    'don', 'doesn', 'didn', 'couldn', 'wouldn', 'shouldn',
    'haven', 'hasn', 'hadn', 'isn', 'aren', 'wasn', 'weren',
    'mustn', 'needn', 'shan', 'mightn',
    # Spam call-to-action words.  These are NLTK stopwords but make up
    # half of phishing vocabulary: "click here", "claim now", "this link",
    # "buy now", "tap here", "act now", "available now".  Removing them
    # turns a sentence like "click this link" into just "click link".
    'here', 'now', 'this',
}
STOP_WORDS  = set(stopwords.words('english')) - _KEEP_WORDS
lemmatizer  = WordNetLemmatizer()

# Leetspeak normalization — undo common digit/symbol substitutions when they
# appear *between letters* in a word.  This catches obfuscations like
# "b4nk" → "bank", "acc0unt" → "account", "cl1ck" → "click",
# "m0n3y" → "money", "p@ssword" → "password".
#
# We require the digit/symbol to be surrounded by lowercase letters, so:
#   • pure numbers ("1000")           → untouched
#   • ordinals ("21st", "3rd")        → untouched
#   • SMS slang ("4u", "gr8", "b4")   → untouched (slang dict handles them)
# This step runs BEFORE punctuation removal so "@" and "$" are still present.
_LEET_MAP = {'0':'o','1':'i','3':'e','4':'a','5':'s','7':'t','@':'a','$':'s'}
_leet_pat = re.compile(r'(?<=[a-z])([01345@$])(?=[a-z])')

def _deleet(text: str) -> str:
    # Apply iteratively because each pass can expose new letter–leet–letter
    # neighborhoods (e.g. "pa$$word" → "pa$sword" → "password").
    prev = None
    while prev != text:
        prev = text
        text = _leet_pat.sub(lambda m: _LEET_MAP[m.group(1)], text)
    return text


def preprocess(text: str, normalize_slang: bool = True) -> str:
    """Full pipeline (Section 3.1 of the paper)."""
    # 1. Case folding
    text = text.lower()
    # 2. Entity replacement
    text = re.sub(r'https?://\S+|www\.\S+', ' urltoken ', text)
    text = re.sub(r'\b(?:\+?\d[\d\s\-().]{7,})\b', ' phonetoken ', text)
    # Currency amounts (e.g. "$1,000", "£50", "€99.99") → moneytoken
    text = re.sub(r'[$£€¥]\s?\d[\d,]*(?:\.\d+)?', ' moneytoken ', text)
    # 3. Leetspeak de-obfuscation (before punctuation removal so @ and $ survive)
    text = _deleet(text)
    # 4. Contraction expansion
    text = contractions.fix(text)
    # 5. Punctuation removal (keep alphanumeric)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # 6. Slang normalization (optional)
    if normalize_slang:
        text = _slang_pattern.sub(lambda m: SLANG_DICT[m.group(0)], text)
    # 7. Tokenisation
    tokens = text.split()
    # 8. Stopword removal; standalone digits → `numtoken` so the model keeps a
    #    "this message contains a number" signal instead of losing it entirely.
    tokens = [
        ('numtoken' if t.isdigit() else t)
        for t in tokens
        if t not in STOP_WORDS
    ]
    # 9. Lemmatisation
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)


# Sanity check
samples = [
    "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Txt FA to 87121",
    "U dun say so early hor... U c already then say...",
    "Congrats! Ur the winner of gr8 prize! Claim ur reward at http://win.com",
    "Congratulations! You have won a $1,000 Walmart gift card. Click here to claim now!",
    "URGENT! Your b4nk acc0unt has been suspended. Cl1ck here to ver1fy.",
]
print('=== Preprocessing Sanity Check ===')
for s in samples:
    print(f'ORIGINAL  : {s}')
    print(f'NO  NORM  : {preprocess(s, normalize_slang=False)}')
    print(f'WITH NORM : {preprocess(s, normalize_slang=True)}')
    print()


In [ ]:
# ── Cell 6: Build DataFrame, EDA & preprocessing ─────────────────────────────
# Convert the two Hugging Face splits into one DataFrame, keeping a 'split'
# column so we can honour the dataset's official train/test partition (Cell 7).
df_train = hf_ds['train'].to_pandas()[['label', 'text']].copy()
df_test  = hf_ds['test'].to_pandas()[['label', 'text']].copy()
df_train['split'] = 'train'
df_test['split']  = 'test'
df = pd.concat([df_train, df_test], ignore_index=True)
df.dropna(subset=['label', 'text'], inplace=True)
df = df[df['text'].str.strip().astype(bool)].reset_index(drop=True)  # drop empty texts

n_train = int((df['split'] == 'train').sum())
n_test  = int((df['split'] == 'test').sum())
print(f'Dataset: {df.shape[0]} rows  (train={n_train}, test={n_test})')
print(df['label'].value_counts().to_string())

# EDA plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Class Distribution')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')
df['text_len'] = df['text'].apply(len)
for label, color in [('not_spam', 'steelblue'), ('spam', 'tomato')]:
    axes[1].hist(df[df['label'] == label]['text_len'], bins=40, alpha=0.6, color=color, label=label)
axes[1].set_title('Message Length Distribution')
axes[1].set_xlabel('Characters')
axes[1].set_xlim(0, df['text_len'].quantile(0.99))   # clip extreme outliers (texts up to ~41k chars)
axes[1].legend()
plt.tight_layout()
plt.savefig('eda_distribution.png', dpi=150)
plt.show()

print('\nApplying preprocessing (a few minutes on ~10.9k rows)...')
t0 = time.time()
df['proc_no_norm']   = df['text'].apply(lambda x: preprocess(x, normalize_slang=False))
df['proc_with_norm'] = df['text'].apply(lambda x: preprocess(x, normalize_slang=True))
print(f'Done in {time.time()-t0:.1f}s')

le = LabelEncoder()
y  = le.fit_transform(df['label'])   # not_spam=0, spam=1 (alphabetical → spam is positive class)
print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_).astype(int)))}')


In [ ]:
# ── Cell 7: TF-IDF feature matrices (official train/test split) ──────────────
NGRAM_CONFIGS = {
    'Unigram': (1, 1),
    'Bigram':  (1, 2),
    'Trigram': (1, 3),
}
NORM_CONFIGS = {
    'No Normalization':   df['proc_no_norm'],
    'With Normalization': df['proc_with_norm'],
}

# Use the dataset's official train/test partition (the 'split' column from Cell 6)
idx_train = df.index[df['split'] == 'train'].to_numpy()
idx_test  = df.index[df['split'] == 'test'].to_numpy()
y_train, y_test = y[idx_train], y[idx_test]
print(f'Train: {len(idx_train)} | Test: {len(idx_test)}')
print(f'Train spam ratio: {y_train.mean():.3f} | Test spam ratio: {y_test.mean():.3f}')

feature_matrices = {}
for norm_name, corpus in NORM_CONFIGS.items():
    for ngram_name, ngram_range in NGRAM_CONFIGS.items():
        vec  = TfidfVectorizer(ngram_range=ngram_range, max_features=10000, sublinear_tf=True)
        X_tr = vec.fit_transform(corpus.iloc[idx_train])
        X_te = vec.transform(corpus.iloc[idx_test])
        feature_matrices[(norm_name, ngram_name)] = (X_tr, X_te, vec)
        print(f'[{norm_name} | {ngram_name}]  vocab={len(vec.vocabulary_):,}')

print('Feature matrices ready.')


In [ ]:
# ── Cell 8: Model definitions & evaluation helper ─────────────────────────────
# The UCI SMS Spam dataset is imbalanced (ham:spam ≈ 6.5:1).  Without telling
# the boosters about this, they over-fit to "ham" and miss real spam at
# inference time.  Compute scale_pos_weight from the actual training labels.
SCALE_POS_WEIGHT = float((y_train == 0).sum()) / max(1, int((y_train == 1).sum()))
print(f'scale_pos_weight = {SCALE_POS_WEIGHT:.3f}  (ham/spam ratio in train set)')


def get_models():
    return {
        'AdaBoost': AdaBoostClassifier(
            estimator=DecisionTreeClassifier(max_depth=1, class_weight='balanced'),
            n_estimators=100, learning_rate=1.0, random_state=RANDOM_STATE
        ),
        'XGBoost': xgb.XGBClassifier(
            n_estimators=100, max_depth=6, learning_rate=0.1,
            eval_metric='logloss', random_state=RANDOM_STATE, verbosity=0,
            scale_pos_weight=SCALE_POS_WEIGHT,
        ),
        'LightGBM': lgb.LGBMClassifier(
            n_estimators=100, num_leaves=31, learning_rate=0.1,
            random_state=RANDOM_STATE, verbose=-1,
            scale_pos_weight=SCALE_POS_WEIGHT,
        ),
        'CatBoost': CatBoostClassifier(
            iterations=100, depth=6, learning_rate=0.1,
            random_seed=RANDOM_STATE, verbose=0,
            auto_class_weights='Balanced',
        ),
    }


def evaluate(model, X_tr, X_te, y_tr, y_te):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = round(time.time() - t0, 2)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]
    return {
        'Precision':      precision_score(y_te, y_pred, zero_division=0),
        'Recall':         recall_score(y_te, y_pred, zero_division=0),
        'F1-Score':       f1_score(y_te, y_pred, zero_division=0),
        'ROC-AUC':        roc_auc_score(y_te, y_prob),
        'Train Time (s)': train_time,
        '_y_pred': y_pred,
        '_y_prob': y_prob,
    }

print('Model helpers ready.')


In [ ]:
# ── Cell 9: Run all 24 experiments + save trained artifacts ──────────────────
import os, pickle
os.makedirs('models', exist_ok=True)

results        = []    # flat list → DataFrame
raw_results    = {}    # (norm, ngram, model) → result dict (predictions only)
trained_models = {}    # (norm, ngram, model) → fitted estimator

total = len(NORM_CONFIGS) * len(NGRAM_CONFIGS) * len(get_models())
done  = 0

for norm_name in NORM_CONFIGS:
    for ngram_name in NGRAM_CONFIGS:
        X_tr, X_te, vec = feature_matrices[(norm_name, ngram_name)]
        # Save the fitted vectorizer once per config
        vec_slug = f"{norm_name.replace(' ', '_')}_{ngram_name}"
        with open(f'models/vec_{vec_slug}.pkl', 'wb') as f:
            pickle.dump(vec, f)
        for model_name, model in get_models().items():
            done += 1
            print(f'[{done:02d}/{total}] {norm_name} | {ngram_name} | {model_name} ... ', end='', flush=True)
            res = evaluate(model, X_tr, X_te, y_train, y_test)
            raw_results[(norm_name, ngram_name, model_name)] = res
            trained_models[(norm_name, ngram_name, model_name)] = model
            # Pickle the trained model for the Streamlit app
            with open(f'models/model_{vec_slug}_{model_name}.pkl', 'wb') as f:
                pickle.dump(model, f)
            results.append({
                'Normalization':   norm_name,
                'N-gram':          ngram_name,
                'Model':           model_name,
                'Precision':       res['Precision'],
                'Recall':          res['Recall'],
                'F1-Score':        res['F1-Score'],
                'ROC-AUC':         res['ROC-AUC'],
                'Train Time (s)':  res['Train Time (s)'],
            })
            print(f"F1={res['F1-Score']:.4f}  AUC={res['ROC-AUC']:.4f}  time={res['Train Time (s)']}s")

results_df = pd.DataFrame(results)
results_df.to_csv('experiment_results.csv', index=False)
print('\nAll experiments done. Results saved to experiment_results.csv')
print(f'Trained models saved under: models/ ({len(os.listdir("models"))} files)')

In [ ]:
# ── Cell 10: Full results table ───────────────────────────────────────────────
display_df = results_df.copy()
for col in ['Precision','Recall','F1-Score','ROC-AUC']:
    display_df[col] = display_df[col].map('{:.4f}'.format)
pd.set_option('display.max_rows', 50)
display(display_df)

In [ ]:
# ── Cell 11: Heatmap – F1-Score ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, norm in zip(axes, NORM_CONFIGS):
    sub   = results_df[results_df['Normalization'] == norm]
    pivot = sub.pivot(index='Model', columns='N-gram', values='F1-Score')[['Unigram','Bigram','Trigram']]
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax,
                vmin=results_df['F1-Score'].min()*0.97, vmax=results_df['F1-Score'].max()*1.01)
    ax.set_title(f'F1-Score — {norm}', fontsize=13)
plt.suptitle('F1-Score Heatmap: Models × N-gram × Normalization', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('heatmap_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 12: Heatmap – ROC-AUC ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, norm in zip(axes, NORM_CONFIGS):
    sub   = results_df[results_df['Normalization'] == norm]
    pivot = sub.pivot(index='Model', columns='N-gram', values='ROC-AUC')[['Unigram','Bigram','Trigram']]
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='Blues', ax=ax,
                vmin=results_df['ROC-AUC'].min()*0.97, vmax=1.0)
    ax.set_title(f'ROC-AUC — {norm}', fontsize=13)
plt.suptitle('ROC-AUC Heatmap: Models × N-gram × Normalization', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('heatmap_auc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 13: Grouped bar – normalization impact per N-gram ────────────────────
models_list = list(get_models().keys())
x = np.arange(len(models_list))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
for ax, ngram_name in zip(axes, NGRAM_CONFIGS):
    f1_no   = [results_df[(results_df['Normalization']=='No Normalization') &
                           (results_df['N-gram']==ngram_name) &
                           (results_df['Model']==m)]['F1-Score'].values[0] for m in models_list]
    f1_with = [results_df[(results_df['Normalization']=='With Normalization') &
                           (results_df['N-gram']==ngram_name) &
                           (results_df['Model']==m)]['F1-Score'].values[0] for m in models_list]
    b1 = ax.bar(x - width/2, f1_no,   width, label='No Norm',   color='steelblue', alpha=0.85)
    b2 = ax.bar(x + width/2, f1_with, width, label='With Norm', color='tomato',    alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(models_list, rotation=15)
    ax.set_ylabel('F1-Score'); ax.set_title(f'F1-Score: {ngram_name}')
    ax.set_ylim(max(0, min(f1_no + f1_with) - 0.05), 1.02)
    ax.legend()
    for bar in b1 + b2:
        ax.annotate(f'{bar.get_height():.3f}',
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 2), textcoords='offset points', ha='center', fontsize=7)
plt.suptitle('F1-Score: Impact of Slang Normalization per N-gram & Model', fontsize=13)
plt.tight_layout()
plt.savefig('bar_normalization_impact.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 14: Confusion matrices (best config) ─────────────────────────────────
best_row   = results_df.loc[results_df['F1-Score'].idxmax()]
best_norm  = best_row['Normalization']
best_ngram = best_row['N-gram']
print(f'Best config → {best_norm} | {best_ngram} | {best_row["Model"]}  (F1={best_row["F1-Score"]:.4f})')

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, model_name in zip(axes, models_list):
    y_pred = raw_results[(best_norm, best_ngram, model_name)]['_y_pred']
    cm     = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=le.classes_).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{model_name}', fontsize=11)
plt.suptitle(f'Confusion Matrices — {best_norm} | {best_ngram}', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 15: ROC curves (best config) ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colors  = ['steelblue', 'tomato', 'green', 'purple']
for model_name, color in zip(models_list, colors):
    y_prob = raw_results[(best_norm, best_ngram, model_name)]['_y_prob']
    auc    = raw_results[(best_norm, best_ngram, model_name)]['ROC-AUC']
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.4f})', color=color, lw=2)
ax.plot([0,1],[0,1],'k--', lw=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title(f'ROC Curves — {best_norm} | {best_ngram}')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()

In [ ]:
# ── Cell 16: Training time heatmap ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, norm_name in zip(axes, NORM_CONFIGS):
    sub   = results_df[results_df['Normalization'] == norm_name]
    pivot = sub.pivot(index='Model', columns='N-gram', values='Train Time (s)')[['Unigram','Bigram','Trigram']]
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Greens', ax=ax)
    ax.set_title(f'Training Time (s) — {norm_name}')
plt.suptitle('Training Time Heatmap (seconds)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('heatmap_traintime.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 17: Normalization delta analysis ─────────────────────────────────────
delta_rows = []
for ngram_name in NGRAM_CONFIGS:
    for model_name in models_list:
        f1_no   = results_df[(results_df['Normalization']=='No Normalization') &
                              (results_df['N-gram']==ngram_name) &
                              (results_df['Model']==model_name)]['F1-Score'].values[0]
        f1_with = results_df[(results_df['Normalization']=='With Normalization') &
                              (results_df['N-gram']==ngram_name) &
                              (results_df['Model']==model_name)]['F1-Score'].values[0]
        delta = f1_with - f1_no
        delta_rows.append({'N-gram': ngram_name, 'Model': model_name,
                           'F1 (No Norm)':   round(f1_no, 4),
                           'F1 (With Norm)': round(f1_with, 4),
                           'Delta':          round(delta, 4),
                           'Direction':      '▲ Improved' if delta > 0 else ('▼ Degraded' if delta < 0 else '= Same')})

delta_df = pd.DataFrame(delta_rows)
print('=== F1 Delta: With Normalization vs No Normalization ===')
display(delta_df)
print(f'\nAverage delta across all experiments: {delta_df["Delta"].mean():+.4f}')

In [ ]:
# ── Cell 18: Top-10 & final summary ──────────────────────────────────────────
print('=== Top 10 Configurations by F1-Score ===')
top10 = results_df.nlargest(10, 'F1-Score')[['Normalization','N-gram','Model','Precision','Recall','F1-Score','ROC-AUC','Train Time (s)']].copy()
for col in ['Precision','Recall','F1-Score','ROC-AUC']:
    top10[col] = top10[col].map('{:.4f}'.format)
display(top10.reset_index(drop=True))

print('\n=== Best Configuration per Algorithm ===')
best_per = results_df.loc[results_df.groupby('Model')['F1-Score'].idxmax()]\
    [['Model','Normalization','N-gram','F1-Score','ROC-AUC']].sort_values('F1-Score', ascending=False)
display(best_per.reset_index(drop=True))

print('\n=== Success Criteria (Section 5) ===')
baseline = results_df[(results_df['Normalization']=='No Normalization') &
                       (results_df['N-gram']=='Unigram')]['F1-Score'].mean()
best_f1  = results_df['F1-Score'].max()
print(f'  Baseline avg F1 (No Norm / Unigram) : {baseline:.4f}')
print(f'  Best F1 overall                     : {best_f1:.4f}')
print(f'  Improvement                         : {best_f1-baseline:+.4f}')

print('\n=== FP / FN counts (best config) ===')
fp_rows = []
for model_name in models_list:
    y_pred = raw_results[(best_norm, best_ngram, model_name)]['_y_pred']
    cm = confusion_matrix(y_test, y_pred)
    fp_rows.append({'Model': model_name, 'FP (NotSpam→Spam)': cm[0][1], 'FN (Spam→NotSpam)': cm[1][0]})
display(pd.DataFrame(fp_rows))


In [ ]:
# ── Cell 19: Download all output files ───────────────────────────────────────
from google.colab import files as colab_files
import os

outputs = [
    'experiment_results.csv',
    'eda_distribution.png',
    'heatmap_f1.png',
    'heatmap_auc.png',
    'bar_normalization_impact.png',
    'confusion_matrices.png',
    'roc_curves.png',
    'heatmap_traintime.png',
]

for f in outputs:
    if os.path.exists(f):
        colab_files.download(f)
        print(f'Downloaded: {f}')
    else:
        print(f'Not found (skipped): {f}')

---
## 🌐 Web UI — Streamlit Demo (Optional)

Run the cells below to launch an interactive web app for testing the trained models.

**What it does:**
- Lets users paste an SMS message
- Toggles for slang normalization & N-gram choice
- Shows predictions from **all 4 boosting models side-by-side**
- Displays preprocessing trace + consensus verdict

**Run these in order:** Cell 20 writes `app.py` → Cell 21 installs Streamlit + tunnel → Cell 22 launches the app.

In [ ]:
%%writefile app.py
"""
SMS Spam Detector — Streamlit Web UI
Compares AdaBoost, XGBoost, LightGBM, CatBoost predictions side-by-side.

Run locally:   streamlit run app.py
Run in Colab:  see the notebook's launcher cell
"""
import os
import re
import pickle
import streamlit as st
import contractions
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet',   quiet=True)
nltk.download('omw-1.4',   quiet=True)

# ─── Slang dictionary (must match training) ──────────────────────────────────
SLANG_DICT = {
    "u": "you", "ur": "your", "r": "are", "b": "be", "y": "why", "k": "ok",
    "m": "am", "n": "and", "2": "to", "4": "for", "4u": "for you", "b4": "before",
    "2day": "today", "2nite": "tonight", "2moro": "tomorrow", "2morrow": "tomorrow",
    "2mrw": "tomorrow", "4got": "forgot", "4ever": "forever", "2gether": "together",
    "gm": "good morning", "gn": "good night", "gud": "good", "gr8": "great",
    "g8": "great", "gr8t": "great", "lol": "laugh out loud", "lmao": "laugh my ass off",
    "rofl": "rolling on the floor laughing", "omg": "oh my god", "omfg": "oh my god",
    "wtf": "what the hell", "wth": "what the hell", "idk": "i do not know",
    "imo": "in my opinion", "imho": "in my humble opinion", "tbh": "to be honest",
    "ngl": "not gonna lie", "fyi": "for your information", "btw": "by the way",
    "jk": "just kidding", "brb": "be right back", "bbl": "be back later",
    "bbs": "be back soon", "afk": "away from keyboard", "np": "no problem",
    "yw": "you are welcome", "ty": "thank you", "thx": "thanks", "thnx": "thanks",
    "tnx": "thanks", "tx": "thanks", "pls": "please", "plz": "please", "plx": "please",
    "sry": "sorry", "srry": "sorry", "nvm": "never mind", "nvr": "never", "evr": "ever",
    "wut": "what", "wat": "what", "wats": "what is", "abt": "about", "hw": "how",
    "hru": "how are you", "wen": "when", "whr": "where", "wer": "where",
    "der": "there", "dere": "there", "dis": "this", "dat": "that", "dats": "that is",
    "d": "the", "da": "the", "de": "the", "dem": "them", "dey": "they",
    "asap": "as soon as possible", "atm": "at the moment", "nite": "night",
    "nyt": "night", "tmrw": "tomorrow", "tmr": "tomorrow", "tmo": "tomorrow",
    "l8r": "later", "l8": "late", "ltr": "later", "ttyl": "talk to you later",
    "ttys": "talk to you soon", "tc": "take care", "cya": "see you", "cu": "see you",
    "cul8r": "see you later", "b4n": "bye for now", "wanna": "want to",
    "gonna": "going to", "gotta": "got to", "kinda": "kind of", "sorta": "sort of",
    "dunno": "do not know", "lemme": "let me", "gimme": "give me", "cmon": "come on",
    "w8": "wait", "h8": "hate", "m8": "mate", "d8": "date", "f8": "fate",
    "sk8": "skate", "ne1": "anyone", "ne": "any", "neway": "anyway",
    "nething": "anything", "evry1": "everyone", "every1": "everyone",
    "som1": "someone", "sum1": "someone", "no1": "no one", "txt": "text",
    "msg": "message", "msgs": "messages", "mob": "mobile", "mobi": "mobile",
    "rply": "reply", "rpl": "reply", "stp": "stop", "dont": "do not",
    "doesnt": "does not", "cant": "cannot", "wont": "will not", "couldnt": "could not",
    "wouldnt": "would not", "shouldnt": "should not", "im": "i am", "ive": "i have",
    "id": "i would", "ill": "i will", "isnt": "is not", "arent": "are not",
    "wasnt": "was not", "werent": "were not", "havent": "have not", "hasnt": "has not",
    "hadnt": "had not", "didnt": "did not", "theyre": "they are", "theyd": "they would",
    "theyll": "they will", "theyve": "they have", "youre": "you are", "youd": "you would",
    "youll": "you will", "youve": "you have", "hes": "he is", "shes": "she is",
    "its": "it is", "weve": "we have", "wed": "we would", "wk": "week",
    "wkly": "weekly", "mth": "month", "yr": "year", "min": "minute", "mins": "minutes",
    "hr": "hour", "hrs": "hours", "wks": "weeks", "acc": "account", "acct": "account",
    "amt": "amount", "cd": "could", "gt": "got", "hav": "have", "hve": "have",
    "nd": "and", "sm": "some", "sme": "some", "ppl": "people", "nt": "not",
    "nw": "now", "nxt": "next", "smth": "something", "sumthing": "something",
    "grt": "great", "amzing": "amazing", "cust": "customer", "custmr": "customer",
    "dlr": "dollar", "dlrs": "dollars", "reciev": "receive", "recive": "receive",
    "receve": "receive", "reedem": "redeem", "redeme": "redeem",
    "subscrib": "subscribe", "unsubscrib": "unsubscribe", "exclusiv": "exclusive",
    "xclusiv": "exclusive", "mrkt": "market", "mkt": "market", "priz": "prize",
}

_slang_keys_sorted = sorted(SLANG_DICT.keys(), key=len, reverse=True)
_slang_pattern = re.compile(
    r'\b(' + '|'.join(re.escape(k) for k in _slang_keys_sorted) + r')\b'
)

# Keep spam-signal words that NLTK's stopword list strips (won, not, no,
# and de-apostrophised negation stems like couldn/wouldn/haven).
_KEEP_WORDS = {
    'won', 'win', 'no', 'not', 'only', 'very', 'against', 'again',
    'don', 'doesn', 'didn', 'couldn', 'wouldn', 'shouldn',
    'haven', 'hasn', 'hadn', 'isn', 'aren', 'wasn', 'weren',
    'mustn', 'needn', 'shan', 'mightn',
    # Spam call-to-action words (NLTK strips them; phishing relies on them).
    'here', 'now', 'this',
}
STOP_WORDS = set(stopwords.words('english')) - _KEEP_WORDS
lemmatizer = WordNetLemmatizer()

# Leetspeak normalization — see notebook Cell 5 for rationale.
_LEET_MAP = {'0':'o','1':'i','3':'e','4':'a','5':'s','7':'t','@':'a','$':'s'}
_leet_pat = re.compile(r'(?<=[a-z])([01345@$])(?=[a-z])')

def _deleet(text: str) -> str:
    prev = None
    while prev != text:
        prev = text
        text = _leet_pat.sub(lambda m: _LEET_MAP[m.group(1)], text)
    return text


def preprocess(text: str, normalize_slang: bool = True) -> str:
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' urltoken ', text)
    text = re.sub(r'\b(?:\+?\d[\d\s\-().]{7,})\b', ' phonetoken ', text)
    # Currency amounts (e.g. "$1,000", "£50") → moneytoken
    text = re.sub(r'[$£€¥]\s?\d[\d,]*(?:\.\d+)?', ' moneytoken ', text)
    text = _deleet(text)   # undo b4nk / acc0unt / p@ssword style obfuscations
    text = contractions.fix(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    if normalize_slang:
        text = _slang_pattern.sub(lambda m: SLANG_DICT[m.group(0)], text)
    tokens = text.split()
    # Keep numbers as a signal: standalone digits → `numtoken`.
    tokens = [
        ('numtoken' if t.isdigit() else t)
        for t in tokens
        if t not in STOP_WORDS
    ]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)


# ─── Model loading (cached) ───────────────────────────────────────────────────
@st.cache_resource
def load_artifact(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


MODELS_DIR = 'models'
MODEL_NAMES = ['AdaBoost', 'XGBoost', 'LightGBM', 'CatBoost']
MODEL_COLORS = {'AdaBoost': '#1f77b4', 'XGBoost': '#d62728',
                'LightGBM': '#2ca02c', 'CatBoost': '#9467bd'}


# ─── Page setup ───────────────────────────────────────────────────────────────
st.set_page_config(page_title='SMS Spam Detector', page_icon='📩', layout='wide')
st.title('📩 SMS Spam Detector')
st.caption('Boosting model comparison • AdaBoost / XGBoost / LightGBM / CatBoost')
st.markdown('---')


# ─── Sidebar configuration ────────────────────────────────────────────────────
with st.sidebar:
    st.header('⚙️ Configuration')
    normalize = st.checkbox('Use slang normalization', value=True,
                            help='Applies the SMS slang lexicon (e.g., "u" → "you").')
    ngram = st.selectbox('N-gram range', ['Unigram', 'Bigram', 'Trigram'],
                         help='Unigram = single words. Bigram/Trigram include 2- and 3-word phrases.')
    st.markdown('---')
    st.markdown('### About')
    st.markdown(
        'Evaluating the impact of **slang normalization** and **N-gram features** '
        'on boosting models for SMS spam detection.\n\n'
        '**Dataset:** UCI SMS Spam Collection (5,572 messages)\n\n'
        '**Authors:**\n- Alexander C. S. Linggodigdo\n- Fanny O. Pangestu\n- Howard F. Goh'
    )


# ─── Build artifact keys ──────────────────────────────────────────────────────
norm_slug = 'With_Normalization' if normalize else 'No_Normalization'
config_slug = f'{norm_slug}_{ngram}'
vec_path = os.path.join(MODELS_DIR, f'vec_{config_slug}.pkl')

if not os.path.exists(vec_path):
    st.error(
        f'❌ Trained models not found at `{MODELS_DIR}/`. '
        'Run the training notebook (`spam_detection_colab.ipynb`) first '
        'and make sure the `models/` folder is in the same directory as `app.py`.'
    )
    st.stop()


# ─── Input area ───────────────────────────────────────────────────────────────
st.subheader('✉️ Enter an SMS message')

EXAMPLES = {
    '— Try a sample —': '',
    'Sample (spam): Free prize':
        'Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Txt FA to 87121 to receive entry',
    'Sample (spam): Claim winner':
        "Congrats! Ur the winner of a gr8 1000 dollar prize! Claim ur reward now at http://win-now.example",
    'Sample (ham): Casual':
        'Hey, are you coming to the party tonight? Let me know if you need a ride.',
    'Sample (ham): Slang':
        'lol u r so funny, ttyl gotta finish hw first',
}

example_choice = st.selectbox('Quick samples:', list(EXAMPLES.keys()))
default_text = EXAMPLES[example_choice]

text = st.text_area('Message:', value=default_text, height=120,
                    placeholder='Paste an SMS message here…')

predict_clicked = st.button('🔍 Predict', type='primary')


# ─── Run prediction ───────────────────────────────────────────────────────────
if predict_clicked:
    if not text.strip():
        st.warning('Please enter a message first.')
        st.stop()

    processed = preprocess(text, normalize_slang=normalize)

    # Preprocessing trace
    st.markdown('### 🔧 Preprocessing Trace')
    c1, c2 = st.columns(2)
    with c1:
        st.markdown('**Original input**')
        st.code(text, language=None)
    with c2:
        st.markdown(f'**After preprocessing** ({norm_slug.replace("_", " ")}, {ngram})')
        st.code(processed if processed.strip() else '(empty — message contained only stopwords/punctuation)',
                language=None)

    # Vectorize
    vec = load_artifact(vec_path)
    X = vec.transform([processed])

    if X.nnz == 0:
        st.warning(
            '⚠️ The preprocessed text contains no vocabulary the models have seen. '
            'Predictions may be unreliable.'
        )

    # Show per-model predictions
    st.markdown('### 🤖 Model Predictions')
    cols = st.columns(4)

    spam_count = 0
    probs = {}
    for col, model_name in zip(cols, MODEL_NAMES):
        model_path = os.path.join(MODELS_DIR, f'model_{config_slug}_{model_name}.pkl')
        model = load_artifact(model_path)
        prob_spam = float(model.predict_proba(X)[0, 1])
        pred = int(prob_spam >= 0.5)
        probs[model_name] = prob_spam
        if pred == 1:
            spam_count += 1

        with col:
            color = MODEL_COLORS[model_name]
            st.markdown(
                f"<div style='border-left: 4px solid {color}; padding-left: 12px;'>"
                f"<h4 style='margin-bottom: 4px;'>{model_name}</h4></div>",
                unsafe_allow_html=True
            )
            if pred == 1:
                st.markdown(
                    "<h2 style='color: #d62728; margin-top: 8px;'>🔴 SPAM</h2>",
                    unsafe_allow_html=True
                )
            else:
                st.markdown(
                    "<h2 style='color: #2ca02c; margin-top: 8px;'>🟢 HAM</h2>",
                    unsafe_allow_html=True
                )
            st.progress(prob_spam)
            st.metric(label='Spam probability', value=f'{prob_spam*100:.2f}%')
            st.caption(f'Ham probability: {(1-prob_spam)*100:.2f}%')

    # Consensus
    st.markdown('### 📊 Model Consensus')
    if spam_count == 4:
        st.error(f'⚠️ **Strong spam signal** — all 4 models flagged this message as SPAM.')
    elif spam_count == 0:
        st.success(f'✅ **Clean message** — all 4 models classified this as HAM.')
    else:
        st.warning(
            f'⚖️ **Mixed verdict** — {spam_count} of 4 models flagged this as SPAM. '
            'Models disagree; the message is borderline.'
        )

    # Probability summary table
    import pandas as pd
    summary = pd.DataFrame({
        'Model': MODEL_NAMES,
        'Prediction': ['SPAM' if probs[m] >= 0.5 else 'HAM' for m in MODEL_NAMES],
        'Spam %': [f'{probs[m]*100:.2f}%' for m in MODEL_NAMES],
        'Ham %':  [f'{(1-probs[m])*100:.2f}%' for m in MODEL_NAMES],
    })
    st.dataframe(summary, hide_index=True, use_container_width=True)


In [ ]:
# ── Cell 22: Launch Streamlit + Cloudflare Tunnel ────────────────────────────
# Cloudflare's tunnel (cloudflared) is more reliable than localtunnel for
# Streamlit — no password page, no JS module errors, and the public URL
# can be opened by anyone on any device while this cell is running.
#
# After running this cell, look for a line in the output like:
#     https://xxxxxxxx-xxxxxx-xxxx.trycloudflare.com
# Click it (or share it with anyone) to open the Spam Detector web app.
#
# To stop the app:  Runtime → Interrupt execution

import os, subprocess, time

# 1. Install Streamlit
print('[1/4] Installing Streamlit…')
subprocess.run(['pip', 'install', '-q', 'streamlit'], check=True)

# 2. Download cloudflared binary (Linux amd64) — ~25 MB, ~5 sec
print('[2/4] Downloading cloudflared…')
if not os.path.exists('cloudflared'):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', 'cloudflared'
    ], check=True)
    subprocess.run(['chmod', '+x', 'cloudflared'], check=True)

# 3. Kill any previous Streamlit process, then start a fresh one
print('[3/4] Starting Streamlit on port 8501…')
subprocess.run(['pkill', '-f', 'streamlit'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
streamlit_proc = subprocess.Popen(
    ['streamlit', 'run', 'app.py',
     '--server.headless=true',
     '--server.port=8501',
     '--server.enableCORS=false',
     '--server.enableXsrfProtection=false',
     '--browser.gatherUsageStats=false'],
    stdout=open('/content/streamlit.log', 'w'),
    stderr=subprocess.STDOUT,
)
time.sleep(6)  # give Streamlit time to boot

# 4. Launch the Cloudflare tunnel — its public URL streams to stdout
print('[4/4] Launching Cloudflare tunnel…')
print('=' * 70)
print('Watch for a line like:   https://<random-words>.trycloudflare.com')
print('That is your public URL — share it with anyone, opens on any device.')
print('=' * 70)
!./cloudflared tunnel --url http://localhost:8501 --no-autoupdate
